In [ ]:
# ==============================================================
# DISCHARGE BIAS CORRECTION — LINEAR SCALING (LS)
# ==============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import numpy as np
import logging

# -------------------------- PATHS --------------------------
data_folder = Path(r"D:\Timeseries")
shapefile_path = r"D:\Discharge.shp"
coords_file = r"D:\Coordinates.csv"
station_seg_id_file = r"D:\Seg_ids.csv"
output_file = r"D:\OUTPUT\CORRECTED_discharge_LS.gpkg"

# Logging
logging.basicConfig(level=logging.INFO)

# -------------------------- FILE MAPPING --------------------------
file_mapping = {
    "A3 Naro Moru": "A3 Naro moru.csv",
    "A4 Naro Moru": "A4 Naro Moru.csv",
    "A6 Naro Moru": "A6 Naro Moru.csv",
    "A8 Burguret": "A8 Burguret.csv",
    "AB Ontulili": "AB Ontulili.csv"
}

# -------------------------- STEP 1 --------------------------
dfs = []
for station, filename in file_mapping.items():
    csv_file = data_folder / filename
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df['Station'] = station
        dfs.append(df)
    else:
        print(f"Warning: {csv_file} not found for station {station}")

station_data = pd.concat(dfs, ignore_index=True)
station_data['Date'] = pd.to_datetime(station_data['Date'])
station_data['Month'] = station_data['Date'].dt.month

# -------------------------- STEP 2: LS FACTORS --------------------------
correction_factors = {}

for station in file_mapping.keys():
    station_df = station_data[station_data['Station'] == station]

    correction_factors[station] = {}

    for month in range(1, 13):
        valid_data = station_df[
            (station_df['VegDischarge'] > 0.01) &
            (station_df['Obs'].notna()) &
            (station_df['VegDischarge'].notna()) &
            (station_df['Month'] == month)
        ]

        if len(valid_data) > 1:
            mean_obs = valid_data['Obs'].mean()
            mean_sim = valid_data['VegDischarge'].mean()

            factor = mean_obs / mean_sim if mean_sim > 0 else 1.0
            correction_factors[station][month] = factor

            print(f"LS factor for {station}, month {month}: {factor:.3f}")

        else:
            print(f"Warning: Insufficient data for {station}, month {month}")
            correction_factors[station][month] = 1.0

# -------------------------- STEP 3 --------------------------
coords = pd.read_csv(coords_file)
coords['Station'] = coords['Station'].str.strip()

shp = gpd.read_file(shapefile_path)

station_seg_id = pd.read_csv(data_folder / station_seg_id_file)
station_seg_id_map = dict(zip(station_seg_id['Station'], station_seg_id['seg_id']))

shp['station'] = shp['seg_id'].map(
    {v: k for k, v in station_seg_id_map.items()}
)

discharge_cols = [col for col in shp.columns if col.startswith('20') and '_' in col]

# -------------------------- STEP 4: APPLY LS --------------------------
original_shp = shp.copy()

for station, factor_dict in correction_factors.items():
    mask = shp['station'] == station

    if mask.any():
        print(f"Applying LS correction to station {station}")

        for col in discharge_cols:
            month = pd.to_datetime(col, format='%Y_%m').month
            factor = factor_dict.get(month, 1.0)

            corrected_values = shp.loc[mask, col] * factor

            negative_count = (corrected_values < 0).sum()
            if negative_count > 0:
                print(f"Warning: {negative_count} negative values before clipping")

            shp.loc[mask, col] = np.clip(corrected_values, 0, None)

    else:
        print(f"Warning: No segments matched for station {station}")

# -------------------------- STEP 5 (UNCHANGED VALIDATION) --------------------------
metrics_summary = []

for station in file_mapping.keys():
    station_df = station_data[station_data['Station'] == station]

    matching_rows = station_seg_id[station_seg_id['Station'] == station]
    if matching_rows.empty:
        continue

    seg_id = matching_rows['seg_id'].iloc[0]

    station_seg_orig = original_shp[original_shp['seg_id'] == seg_id]
    station_seg_corr = shp[shp['seg_id'] == seg_id]

    discharge_cols = [col for col in station_seg_orig.columns if col.startswith('20') and '_' in col]

    veg_orig = station_seg_orig[discharge_cols].values.flatten()
    veg_corr = station_seg_corr[discharge_cols].values.flatten()
    dates = pd.to_datetime(discharge_cols, format='%Y_%m', errors='coerce')

    csv_dates = station_df['Date'].dt.strftime('%Y_%m')
    geo_date_strs = dates.strftime('%Y_%m')

    mask = geo_date_strs.isin(set(csv_dates))
    geo_indices = np.where(mask)[0]

    obs_series = station_df.set_index('Date')['Obs'].reindex(
        pd.to_datetime(csv_dates, format='%Y_%m')
    ).loc[dates[geo_indices].date].values

    veg_corr_valid = veg_corr[geo_indices]

    valid_mask = (~np.isnan(obs_series)) & (~np.isnan(veg_corr_valid))

    obs_valid = obs_series[valid_mask]
    veg_valid = veg_corr_valid[valid_mask]

    if len(obs_valid) == 0:
        continue

    nse = 1 - np.sum((obs_valid - veg_valid) ** 2) / np.sum((obs_valid - np.mean(obs_valid)) ** 2)
    rmse = np.sqrt(mean_squared_error(obs_valid, veg_valid))
    corr = np.corrcoef(obs_valid, veg_valid)[0, 1]

    metrics_summary.append({
        'Station': station,
        'Seg_ID': seg_id,
        'NSE': nse,
        'RMSE': rmse,
        'Correlation': corr
    })

# Save metrics
pd.DataFrame(metrics_summary).to_csv(data_folder / "validation_metrics_ls.csv", index=False)

# -------------------------- STEP 6  APPLY DISCHARGE-AREA-PROPORTION METHOD TO ESTIMATE MONTHLY DISCHARGES FOR OTHER SEGMENTS--------------------------
print("#Step 6: Apply Discharge-Area-Proportion method")
for station in file_mapping.keys():
    station_shp = shp[shp['station'] == station]
    if station_shp.empty:
        print(f"Warning: No data found for station {station} in shapefile")
        continue
    seg_id = station_shp['seg_id'].iloc[0]
    uparea_station = station_shp['uparea'].iloc[0] if 'uparea' in shp.columns else None
    
    if uparea_station is None:
        print(f"Warning: 'uparea' column not found or missing for station {station}")
        continue

    # Downstream estimation
    if 'NextDownID' not in shp.columns:
        print("Warning: 'NextDownID' column not found for downstream estimation")
    else:
        downstream_id = station_shp['NextDownID'].iloc[0]
        if pd.notna(downstream_id) and downstream_id in shp['seg_id'].values:
            downstream_shp = shp[shp['seg_id'] == downstream_id].copy()
            uparea_down = downstream_shp['uparea'].iloc[0]
            for col in discharge_cols:
                discharge = station_shp[col].iloc[0]
                estimated_discharge = discharge * (uparea_down / uparea_station)
                # Clip negative values to zero for downstream estimation
                if estimated_discharge < 0:
                    print(f"Warning: Negative discharge ({estimated_discharge:.3f}) for {station}, {col} in downstream estimation, clipping to 0")
                shp.loc[shp['seg_id'] == downstream_id, col] = max(estimated_discharge, 0)

    # Upstream estimation
    for up_col in ['up1', 'up2', 'up3', 'up4']:
        if up_col not in shp.columns:
            print(f"Warning: '{up_col}' column not found for upstream estimation")
        else:
            upstream_id = station_shp[up_col].iloc[0]
            if pd.notna(upstream_id) and upstream_id in shp['seg_id'].values:
                upstream_shp = shp[shp['seg_id'] == upstream_id].copy()
                uparea_up = upstream_shp['uparea'].iloc[0]
                for col in discharge_cols:
                    discharge = station_shp[col].iloc[0]
                    estimated_discharge = discharge * (uparea_up / uparea_station)
                    # Clip negative values to zero for upstream estimation
                    if estimated_discharge < 0:
                        print(f"Warning: Negative discharge ({estimated_discharge:.3f}) for {station}, {col} in upstream estimation, clipping to 0")
                    shp.loc[shp['seg_id'] == upstream_id, col] = max(estimated_discharge, 0)


# -------------------------- STEP 7 --------------------------
shp = shp.drop(columns=['station'])
shp.to_file(output_file, driver="GPKG")

print("LS discharge correction complete")